In [ ]:
# =============================================================================
# explore/Explore_exposure_inputs.py
# What can this workspace reach, and what does it already hold, for the exposure and risk grid?
# (plan: docs/EXPOSURE_AND_RISK_GRID_PLAN.md). Read only. Writes nothing. Safe to run any time.
#
# It answers three questions so the loaders can be built from facts, not guesses:
#   1. Which public data sites can the cluster reach? (ONS, Nomis, gov.uk, NHS, CQC, DfE, the EA)
#   2. Which governed tables already look like NaFRA2, flood zones, receptors, critical sites or boundaries?
#   3. What files are already in the volumes?
# Photograph or copy the three result tables back. Nothing here is secret: it is names, sizes and status codes.
#
# Widgets: extra_patterns (more words to look for in table names, comma separated), volumes (comma separated volume paths to list).
# =============================================================================

import os
import sys

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import time
import pandas as pd
import requests

import config
from utils.helpers import get_spark

spark = get_spark()

dbutils.widgets.text("extra_patterns", "", "Extra words to look for in table names (comma separated)")
dbutils.widgets.text("volumes", f"/Volumes/{config.CATALOG}/{config.SCHEMA}/{config.EXTERNAL_VOLUME},/Volumes/{config.CATALOG}/{config.RESTRICTED_SCHEMA}/{config.EXTERNAL_VOLUME}", "Volumes to list (comma separated)")
extra = [w.strip().lower() for w in dbutils.widgets.get("extra_patterns").split(",") if w.strip()]
volumes = [v.strip() for v in dbutils.widgets.get("volumes").split(",") if v.strip()]

In [ ]:
# =============================================================================
# 1. WHICH PUBLIC SITES CAN THIS CLUSTER REACH?
# A short GET with a timeout. 200 or 3xx means reachable. A timeout or a connection error means blocked or down.
# =============================================================================

SITES = [
    ("ONS Open Geography Portal", "https://geoportal.statistics.gov.uk/"),
    ("ONS ArcGIS services", "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services?f=json"),
    ("Nomis Census 2021 bulk table", "https://www.nomisweb.co.uk/output/census/2021/census2021-ts001.zip"),
    ("gov.uk statistics: IoD2025", "https://www.gov.uk/government/statistics/english-indices-of-deprivation-2025"),
    ("gov.uk asset host", "https://assets.publishing.service.gov.uk/"),
    ("data.gov.uk", "https://www.data.gov.uk/"),
    ("EA data portal", "https://environment.data.gov.uk/"),
    ("EA flood monitoring API", "https://environment.data.gov.uk/flood-monitoring/id/floods?_limit=1"),
    ("Defra data services platform", "https://environment.data.gov.uk/dataset"),
    ("NHS ODS reference files", "https://files.digital.nhs.uk/assets/ods/current/epraccur.zip"),
    ("NHS ODS API", "https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations?Limit=1"),
    ("CQC public API", "https://api.service.cqc.org.uk/public/v1/providers?perPage=1"),
    ("DfE Get Information About Schools", "https://get-information-schools.service.gov.uk/Downloads"),
    ("OpenStreetMap Overpass", "https://overpass-api.de/api/status"),
]

rows = []
for name, url in SITES:
    t0 = time.time()
    try:
        r = requests.get(url, timeout=20, stream=True, allow_redirects=True, headers={"User-Agent": "FGS-exposure-survey"})
        rows.append({"site": name, "reachable": r.status_code < 400, "status": r.status_code, "type": r.headers.get("Content-Type", "")[:40],
                     "size": r.headers.get("Content-Length", ""), "seconds": round(time.time() - t0, 1), "url": url[:70]})
        r.close()
    except Exception as e:
        rows.append({"site": name, "reachable": False, "status": "", "type": "", "size": "", "seconds": round(time.time() - t0, 1),
                     "url": url[:70], "error": str(e)[:90]})
sites = pd.DataFrame(rows)
print("1. Public sites")
display(sites)

In [ ]:
# =============================================================================
# 2. WHICH TABLES IN THE CATALOGUE LOOK RELEVANT?
# Names only (and the table comment): no data is read. Uses the information schema, so it needs permission to see it.
# =============================================================================

PATTERNS = ["nafra", "rofrs", "rofsw", "risk_of_flood", "flood_zone", "fz2", "fz3", "flood_map", "receptor", "nrd", "properties", "property",
            "aims", "defence", "census", "imd", "deprivation", "lsoa", "output_area", "msoa", "oa_", "boundar", "lrf", "resilience",
            "combined_auth", "local_auth", "lad_", "police", "fire", "hospital", "gp_", "school", "care_home", "cqc", "substation",
            "sewage", "water_treat", "comah", "prison", "critical", "infrastructure", "addressbase", "uprn", "postcode", "onspd"] + extra

pattern = "|".join(PATTERNS)
found = None
try:
    found = spark.sql(f"""
        SELECT table_catalog, table_schema, table_name, table_type, comment
        FROM system.information_schema.tables
        WHERE table_schema <> 'information_schema' AND lower(table_name) RLIKE '{pattern}'
        ORDER BY table_catalog, table_schema, table_name
    """).toPandas()
except Exception as e:
    print("The system information schema is not readable here (" + str(e)[:120] + "). Falling back to the catalogs this workspace can list.")
    rows = []
    for cat in [c.name for c in spark.catalog.listCatalogs()]:
        try:
            for s in spark.sql(f"SHOW SCHEMAS IN `{cat}`").collect():
                sch = s[0]
                if sch == "information_schema":
                    continue
                try:
                    for t in spark.sql(f"SHOW TABLES IN `{cat}`.`{sch}`").collect():
                        if any(p in t["tableName"].lower() for p in PATTERNS):
                            rows.append({"table_catalog": cat, "table_schema": sch, "table_name": t["tableName"], "table_type": "", "comment": ""})
                except Exception:
                    pass
        except Exception:
            pass
    found = pd.DataFrame(rows)
print(f"2. Tables whose names look relevant: {0 if found is None else len(found)}")
display(found if found is not None else pd.DataFrame())

In [ ]:
# =============================================================================
# 3. WHAT IS ALREADY IN THE VOLUMES? (names and sizes, two levels deep)
# =============================================================================

def list_volume(path, depth=2):
    out = []
    try:
        for f in dbutils.fs.ls(path):
            out.append({"path": f.path.replace("dbfs:", ""), "size_mb": round(f.size / 1e6, 2), "is_dir": f.name.endswith("/")})
            if f.name.endswith("/") and depth > 1:
                out.extend(list_volume(f.path, depth - 1))
    except Exception as e:
        out.append({"path": path, "size_mb": None, "is_dir": None, "error": str(e)[:100]})
    return out

files = pd.DataFrame([r for v in volumes for r in list_volume(v)])
print(f"3. Files in the volumes: {len(files)}")
display(files)
print("\nDone. Send the three tables back: sites, tables and files.")
dbutils.notebook.exit("success")